# Run x05: ask a pretrained virtual cell model what happens next

An exploration from the Virtual Cell Lab. We give Arc Institute's **State** model some real K562 control cells and ask it to
predict what five gene knockdowns would do, knockdowns it never trained on in K562. Then we look at what comes back, and ask
what the answer can't tell us. Nothing is scored against measured data here; that is chapter 6, with baselines.

The predictions this notebook checks were committed before it was run: `run-card.yaml` in this folder.

**Every prediction below is model output, not a biological finding.**

> Model outputs from State. Citation required by the State licence: Adduri, A. et al. (2025) *Predicting cellular responses
> to perturbation across diverse contexts with State.* bioRxiv 10.1101/2025.06.26.661135; Cell 2026, doi 10.1016/j.cell.2026.07.052.

Runtime: **Colab with a GPU** (Runtime → Change runtime type → T4 GPU). It does not run on CPU.

## 0. The licence: read this before running anything

State's weights, files and outputs are under the [Arc Research Institute State Model Non-Commercial License](https://github.com/ArcInstitute/state/blob/main/MODEL_LICENSE.md).
Downloading or running the model means you accept it. In short:

- Use is allowed only for **Non-Commercial Purposes**. Research or education "engaged in or sponsored by a Commercial Entity" does not count as non-commercial (section 1.4).
- Outputs may be shared, with a prominent citation to the State paper (section 3).
- No commercial use without a separate written licence from Arc (section 4.1).

This is a summary, not legal advice. If you are not sure your use qualifies, ask Arc (legal@arcinstitute.org) before running.
Set the flag below to `True` only if you have read the licence and your use qualifies.

In [ ]:
ACCEPT_STATE_NONCOMMERCIAL_LICENSE = False  # set to True only after reading the licence above

assert ACCEPT_STATE_NONCOMMERCIAL_LICENSE, "Read the State Model Non-Commercial License and set the flag to True if your use qualifies."

## 1. Install State and check for a GPU

In [ ]:
%pip install -q "arc-state==0.11.1" "huggingface_hub>=0.24"

In [ ]:
import json, pickle, subprocess, time, tomllib
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "No GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then run again."
print(torch.cuda.get_device_name(0))
SEED = 13
results = {"run": "x05-small-inference", "label": "model_output", "gpu": torch.cuda.get_device_name(0)}

## 2. Download the model and the cells, pinned to exact versions

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

MODEL_REPO, MODEL_REV = "arcinstitute/ST-HVG-Replogle", "bb6a9562cbbf1fd152df14cc53b4cc7517c77175"
DATA_REPO, DATA_REV = "arcinstitute/State-Replogle-Filtered", "d790193bb2c93726541a75ca3fa873a92ed44da5"

local = snapshot_download(MODEL_REPO, revision=MODEL_REV, local_dir="state_model", allow_patterns=[
    "fewshot/k562/config.yaml", "fewshot/k562/checkpoints/final.ckpt", "fewshot/k562/*.pkl", "fewshot/k562/*.pt",
    "fewshot/k562/*.torch", "fewshot/k562/eval_best.ckpt/adata_real.h5ad"])
RUN = Path(local) / "fewshot" / "k562"
toml_path = hf_hub_download(DATA_REPO, "k562.toml", repo_type="dataset", revision=DATA_REV, local_dir="state_data")
sorted(p.name for p in RUN.iterdir())

## 3. Look inside before trusting anything

In [ ]:
var_dims = pickle.load(open(RUN / "var_dims.pkl", "rb"))
print({k: (len(v) if hasattr(v, "__len__") and not isinstance(v, str) else v) for k, v in var_dims.items()})
pert_map = torch.load(RUN / "pert_onehot_map.pt", weights_only=False)
vocab = set(map(str, pert_map))
print("perturbations the model knows:", len(vocab))

real = ad.read_h5ad(RUN / "eval_best.ckpt" / "adata_real.h5ad")
print(real)
print(real.obs.head())

In [ ]:
PERT, CTRL, CELLTYPE, BATCH = "gene", "non-targeting", "cell_line", "gem_group"
for col in (PERT, CELLTYPE):
    assert col in real.obs, f"expected column {col!r} in the evaluation file; found {list(real.obs.columns)}"
is_ctrl = real.obs[PERT].astype(str) == CTRL
print("control cells in the file:", int(is_ctrl.sum()))
assert is_ctrl.sum() >= 256, "not enough control cells to sample from"

names = var_dims.get("gene_names")
n_out = real.n_vars
genes = [str(g) for g in names] if names is not None and len(names) == n_out else None
results["output_genes"] = n_out
results["output_genes_named"] = genes is not None
print("output genes:", n_out, "| named:", genes is not None)

## 4. Pick the five knockdowns by the rule on the run card

In [ ]:
test_perts = sorted(tomllib.load(open(toml_path, "rb"))["fewshot"]["replogle.k562"]["test"])
print("held-out K562 knockdowns:", len(test_perts))
pool = [g for g in test_perts if g in vocab and (genes is None or g in genes)]
targets = pool[:5]
UNKNOWN = "NOT_A_REAL_GENE"
assert UNKNOWN not in vocab
results["targets"] = targets
targets

## 5. Build the question: real control cells, labelled with the knockdown we want predicted

In [ ]:
rng = np.random.default_rng(SEED)
ctrl_idx = rng.choice(np.where(is_ctrl)[0], size=256, replace=False)
controls = real[ctrl_idx].copy()
controls.obs = controls.obs.copy()
controls.obs[PERT] = CTRL

blocks = [controls]
for label in targets + [UNKNOWN]:
    pick = rng.choice(controls.n_obs, size=128, replace=True)
    block = controls[pick].copy()
    block.obs = block.obs.copy()
    block.obs[PERT] = label
    block.obs_names = [f"{label}_{i}" for i in range(block.n_obs)]
    blocks.append(block)
query = ad.concat(blocks, axis=0)
query.obs[PERT] = query.obs[PERT].astype(str)
query.write_h5ad("query.h5ad")
query.obs[PERT].value_counts()

## 6. Ask the model

In [ ]:
cmd = ["state", "tx", "infer", "--model-dir", str(RUN), "--checkpoint", str(RUN / "checkpoints" / "final.ckpt"),
       "--adata", "query.h5ad", "--pert-col", PERT, "--output", "predicted.h5ad", "--seed", str(SEED)]
if BATCH in query.obs:
    cmd += ["--batch-col", BATCH]
start = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
results["inference_seconds"] = round(time.time() - start, 1)
results["inference_ok"] = proc.returncode == 0
log = proc.stdout + proc.stderr
print(log[-4000:])
results["unknown_gene_notice"] = list(dict.fromkeys(l.strip() for l in log.splitlines() if UNKNOWN in l))[:3]
assert proc.returncode == 0, "inference failed; the log above says why"

## 7. What came back

In [ ]:
pred = ad.read_h5ad("predicted.h5ad")
X = pred.X.toarray() if hasattr(pred.X, "toarray") else np.asarray(pred.X)
labels = pred.obs[PERT].astype(str).to_numpy()
mean = {lab: X[labels == lab].mean(axis=0) for lab in [CTRL] + targets + [UNKNOWN]}
change = {g: mean[g] - mean[CTRL] for g in targets + [UNKNOWN]}

rows = []
for g in targets:
    row = {"target": g, "size_of_predicted_change": round(float(np.linalg.norm(change[g])), 4)}
    if genes is not None:
        j = genes.index(g)
        row["target_predicted_in_knockdown"] = round(float(mean[g][j]), 4)
        row["target_predicted_in_controls"] = round(float(mean[CTRL][j]), 4)
        row["target_went_down"] = bool(mean[g][j] < mean[CTRL][j])
        top = np.argsort(-np.abs(change[g]))[:10]
        row["top_genes_predicted_to_move"] = [genes[k] for k in top]
    rows.append(row)
results["per_target"] = rows

pairs = [np.corrcoef(change[a], change[b])[0, 1] for i, a in enumerate(targets) for b in targets[i + 1:]]
results["mean_pairwise_correlation_of_changes"] = round(float(np.mean(pairs)), 4)
results["unknown_gene"] = {"returned_rows": int((labels == UNKNOWN).sum()),
                           "size_of_predicted_change": round(float(np.linalg.norm(change[UNKNOWN])), 4),
                           "largest_gene_shift": round(float(np.abs(change[UNKNOWN]).max()), 4)}
pd.DataFrame(rows)

## 8. Check the run card's predictions, and save

In [ ]:
went_down = [r.get("target_went_down") for r in rows]
results["predictions"] = [
    {"statement": "inference under 10 minutes on a T4", "held": bool(results["inference_ok"] and results["inference_seconds"] < 600 and "T4" in results["gpu"])},
    {"statement": "target lower in at least 4 of 5", "held": (sum(went_down) >= 4) if genes is not None else "not evaluable: output genes not named"},
    {"statement": "mean pairwise correlation of changes below 0.9", "held": bool(results["mean_pairwise_correlation_of_changes"] < 0.9)},
    {"statement": "unknown gene returns a prediction with no error", "held": bool(results["inference_ok"] and results["unknown_gene"]["returned_rows"] == 128)},
]
results["citation"] = "Adduri, A. et al. (2025) Predicting cellular responses to perturbation across diverse contexts with State."
Path("results.json").write_text(json.dumps(results, indent=2) + "\n")
print(json.dumps(results["predictions"], indent=2))
print("Saved results.json. Send it back, or download it from the Files panel on the left.")

## What this does not show

- It does not show the predictions are **right**. The measured responses for these knockdowns sit in the same file, unopened on purpose: comparing them is a score, and scores need baselines (chapter 6).
- K562 is a leukemia cell line. The answer says nothing about other cell types, or about tissue.
- It predicts the average shift of a group of cells, not what any single cell will do.
- An unknown gene name still gets an answer. A confident-looking output is not evidence the model knew anything.